In [0]:
%spark.pyspark
from pyspark.sql import functions as F
from pyspark.sql.window import Window

users = spark.createDataFrame(
    [
        ("u1", "Berlin"),
        ("u2", "Berlin"),
        ("u3", "Munich"),
        ("u4", "Hamburg"),
    ],
    ["user_id", "city"]
)

orders = spark.createDataFrame(
    [
        ("o1", "u1", "p1", 2, 10.0),
        ("o2", "u1", "p2", 1, 30.0),
        ("o3", "u2", "p1", 1, 10.0),
        ("o4", "u2", "p3", 5, 7.0),
        ("o5", "u3", "p2", 3, 30.0),
        ("o6", "u3", "p3", 1, 7.0),
        ("o7", "u4", "p1", 10, 10.0),
    ],
    ["order_id", "user_id", "product_id", "qty", "price"]
)

products = spark.createDataFrame(
    [
        ("p1", "Ring VOLA"),
        ("p2", "Ring POROG"),
        ("p3", "Ring TISHINA"),
    ],
    ["product_id", "product_name"]
)

users.show()
orders.show()
products.show()

In [ ]:
%spark.pyspark

orders_with_revenue = orders.withColumn(
    "revenue",
    F.col("qty") * F.col("price")
)

joined_orders = (
    orders_with_revenue
    .join(users, on="user_id", how="inner")
    .join(products, on="product_id", how="inner")
)

joined_orders.show()


In [ ]:
%spark.pyspark

city_product_metrics = (
    joined_orders
    .groupBy("city", "product_id", "product_name")
    .agg(
        F.count("order_id").alias("orders_cnt"),
        F.sum("qty").alias("qty_sum"),
        F.sum("revenue").alias("revenue_sum")
    )
)

city_product_metrics.orderBy("city", F.desc("revenue_sum")).show()

In [ ]:
%spark.pyspark

city_window = Window.partitionBy("city").orderBy(
    F.desc("revenue_sum"),
    F.asc("product_id")
)

mart_city_top_products = (
    city_product_metrics
    .withColumn("rank_in_city", F.row_number().over(city_window))
    .filter(F.col("rank_in_city") <= 2)
    .orderBy("city", "rank_in_city")
)

mart_city_top_products.show()

In [ ]:
%spark.pyspark

hdfs_output_path = "/tmp/sandbox_zeppelin/mart_city_top_products/"

(
    mart_city_top_products
    .write
    .mode("overwrite")
    .parquet(hdfs_output_path)
)

print(f"Data was written to HDFS path: {hdfs_output_path}")

In [ ]:
%spark.pyspark

mart_from_hdfs = spark.read.parquet(hdfs_output_path)

mart_from_hdfs.orderBy("city", "rank_in_city").show()

In [ ]:
%spark.pyspark
s3_output_path = "s3a://hadoop27/mart_city_top_products/"

(
    mart_city_top_products
    .write
    .mode("overwrite")
    .parquet(s3_output_path)
)

print(f"Data was written to S3 path: {s3_output_path}")

In [ ]:
%spark.pyspark
mart_from_s3 = spark.read.parquet(s3_output_path)

mart_from_s3.orderBy("city", "rank_in_city").show()